# ReID — B0 / Seed 2 | 50-Epoch Production

This notebook runs the original pinned research training/evaluation for **B0, seed 2**, using **Kaggle T4 ×2** and all 10 verified REIDPACK3 files.

| Contract | Value |
|---|---|
| Training | **50 epochs × 2,000 steps = 100,000 optimizer steps** |
| Batch | **128 global** (2 DDP ranks) |
| Validation | Every 5 epochs; original V3 evaluation retained |
| Checkpoint policy | Original midpoint / end-epoch writes; automatic verified resume |
| Remote backup | **PRIVATE HF Storage Bucket** `reid-research-hub/reid-loss-study-live` |
| Isolated prefix | `v3_50epoch_28c68a1/B0/seed2/` |
| Large remote files (max 3) | `slot_a.pth`, `slot_b.pth`, `model_best.pth` |
| Historical 60-epoch model repo | **Never read from or written to** |
| Pinned research commit | `28c68a1ea068cbe0ca1ea95273a6f80a5094b6f3` |

**Important:** Hugging Face Buckets overwrite files in place and do *not* create Git commits. Each run has at most three remote `.pth` objects (approximately **1.59 GB** at 530 MB each, plus small JSON/metrics). This limits *new live-checkpoint storage growth*, but does not reclaim the **old 60-epoch Git repository history**. Exact quota display is provider-controlled and cannot be guaranteed to the byte.

**Kaggle prerequisites:** Attach `a7medlabib/reid-fine-tunning-datasets-v3` and enable `GITHUB_TOKEN` (read) + `HF_TOKEN` (organization Bucket write). All remote backups fail closed on checksum/cursor errors. Do not run two sessions of the SAME condition/seed concurrently.

**Critical:** A 50-epoch run has a different config fingerprint and never auto-resumes any old 60-epoch checkpoint. All notebooks must be started with **Save Version → Save & Run All** after selecting T4 ×2.


## 1. Hardware and secrets

Confirm both GPUs, the dataset, and access tokens. Token values are never printed.


In [ ]:
CONDITION = 'B0'
SEED = 2
ENABLE_MID_EPOCH_CHECKPOINT = True
from pathlib import Path
import os
import sys
import time
import torch
from kaggle_secrets import UserSecretsClient

RESEARCH_COMMIT = "28c68a1ea068cbe0ca1ea95273a6f80a5094b6f3"
FASTREID_COMMIT = "c9bc3ceb2f7a6438b62fb515ea3df6d1e999e95d"
REPO_URL = "https://github.com/BASSAT-BASSAT/Person-Re-ID-Foundational-model.git"
HF_BUCKET_ID = "reid-research-hub/reid-loss-study-live"
ROOT = Path("/kaggle/working/Person-Re-ID-Foundational-model-28c68a1")
OUT = Path(f"/kaggle/working/reid_loss_study_50epoch_28c68a1/{CONDITION}/seed{SEED}")
OUT.mkdir(parents=True, exist_ok=True)
SESSION_START = time.time()
os.environ["REIDFM_SESSION_START"] = str(SESSION_START)

pack_candidates = [
    Path("/kaggle/input/datasets/a7medlabib/reid-fine-tunning-datasets-v3"),
    Path("/kaggle/input/reid-fine-tunning-datasets-v3"),
]
PACK = next((p for p in pack_candidates if p.is_dir()), None)
assert PACK is not None, "Attach: a7medlabib/reid-fine-tunning-datasets-v3"
assert len(list(PACK.glob("*.pack"))) == 10, "Expected the 10 V3 dataset packs"
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Select Kaggle GPU T4 x2"
secret_client = UserSecretsClient()
GITHUB_TOKEN = secret_client.get_secret("GITHUB_TOKEN")
HF_TOKEN = secret_client.get_secret("HF_TOKEN")
assert GITHUB_TOKEN, "Enable GITHUB_TOKEN with private GitHub repo read permission"
assert HF_TOKEN, "Enable HF_TOKEN with WRITE permission for the private HF organization Bucket"
print(f"[PASS] {CONDITION} / Seed {SEED} | 2 GPUs | 10 REIDPACK3 files")
print("[PASS] Both Kaggle secrets are available; token values stay hidden")


## 2. Research environment

Pin Git revision and verify critical source hashes before training.


In [ ]:
import json
import hashlib
import importlib
import importlib.util
import subprocess
import tempfile
import collections
import os
import sys
from pathlib import Path


# ============================================================
# 1. Safe command runner
# ============================================================

def checked(args, *, env=None, timeout=900):
    # Never print shell commands or stderr:
    # Git may include authentication context.
    result = subprocess.run(
        args,
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout,
    )

    if result.returncode:
        raise RuntimeError(
            f"Setup command {args[0]!r} failed "
            f"with exit code {result.returncode}"
        )

    return result.stdout.strip()


# ============================================================
# 2. Clone and verify original research repository
# ============================================================

if not ROOT.exists():
    with tempfile.TemporaryDirectory(dir="/kaggle/working") as tmp:
        askpass = Path(tmp) / "askpass.py"

        askpass.write_text(
            "#!/usr/bin/env python3\n"
            "import os,sys\n"
            "p=sys.argv[1].lower() if len(sys.argv)>1 else ''\n"
            "print('x-access-token' if 'username' in p "
            "else os.environ['REID_GITHUB_TOKEN'])\n",
            encoding="utf-8",
        )

        askpass.chmod(0o700)

        git_env = os.environ.copy()
        git_env.update({
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "REID_GITHUB_TOKEN": GITHUB_TOKEN,
        })

        checked(
            ["git", "clone", "--quiet", REPO_URL, str(ROOT)],
            env=git_env,
        )


checked([
    "git", "-C", str(ROOT),
    "checkout", "--quiet", "--detach", RESEARCH_COMMIT
])

assert checked([
    "git", "-C", str(ROOT), "rev-parse", "HEAD"
]) == RESEARCH_COMMIT

print("[PASS] Original research Git commit verified")


# ============================================================
# 3. Clone and verify FastReID
# ============================================================

FASTREID = ROOT / "fast-reid"

if not FASTREID.exists():
    checked([
        "git", "clone", "--quiet",
        "https://github.com/JDAI-CV/fast-reid.git",
        str(FASTREID),
    ])

checked([
    "git", "-C", str(FASTREID),
    "checkout", "--quiet", "--detach", FASTREID_COMMIT
])

assert checked([
    "git", "-C", str(FASTREID), "rev-parse", "HEAD"
]) == FASTREID_COMMIT

print("[PASS] Original FastReID Git commit verified")


# ============================================================
# 4. Install missing dependencies
# ============================================================

requirements = {
    "yacs": "yacs==0.1.8",
    "termcolor": "termcolor>=2",
    "fvcore": "fvcore>=0.1.5",
    "iopath": "iopath>=0.1.9",
    "gdown": "gdown>=5",
    "tensorboard": "tensorboard>=2.14",
}

missing = [
    requirement
    for name, requirement in requirements.items()
    if importlib.util.find_spec(name) is None
]

if missing:
    checked(
        [
            sys.executable, "-m", "pip",
            "install", "--quiet", "--no-input",
            *missing,
        ],
        timeout=600,
    )

if importlib.util.find_spec("faiss") is None:
    checked(
        [
            sys.executable, "-m", "pip",
            "install", "--quiet", "--no-input",
            "--no-deps", "--only-binary=:all:",
            "faiss-cpu==1.15.1",
        ],
        timeout=300,
    )

checked(
    [
        sys.executable, "-m", "pip",
        "install", "--quiet", "--no-input",
        "huggingface_hub>=1.7.1,<2",
    ],
    timeout=600,
)

print("[PASS] Required Python dependencies available")


# ============================================================
# 5. PyTorch / TorchVision import preflight
# ============================================================

try:
    import torch
    import importlib
    import torchvision

    # Import the internal module explicitly.
    # Do not rely on hasattr(torch, "_utils").
    torch_utils = importlib.import_module("torch._utils")

    assert torch_utils is not None, (
        "Failed to import torch._utils"
    )

    assert torch.cuda.is_available(), (
        "CUDA is unavailable in the current Kaggle session"
    )

    assert torch.cuda.device_count() == 2, (
        "This experiment requires exactly 2 GPUs"
    )

    print("[PASS] PyTorch:", torch.__version__)
    print("[PASS] TorchVision:", torchvision.__version__)
    print("[PASS] CUDA:", torch.version.cuda)
    print("[PASS] torch._utils imported successfully")
    print("[PASS] Available GPUs:", torch.cuda.device_count())

except Exception as exc:
    raise RuntimeError(
        "PyTorch/TorchVision import preflight failed. "
        "Inspect the original exception before starting training."
    ) from exc

# ============================================================
# 6. Verify original research source fingerprints
# ============================================================

verified_sources = {
    "reidfm/engine/trainer.py":
        "ce107c2c8d67e7ed06a990ab35a977ade638d4b152f3b82f176f49f64fc904e6",

    "reidfm/engine/v3_freeze_counter.py":
        "8f1d962ca7b0f984c9ae46ac34334b7bf2e9b83d443d9450f61daab7a186dd34",

    "reidfm/data/v3_fastreid_dataset.py":
        "8f7024e2edfe277d8e7c5fd031af845a29fbbde1d37859aaf0754d24d521f849",
}

for name, expected in verified_sources.items():
    actual = hashlib.sha256(
        (ROOT / name).read_bytes()
    ).hexdigest()

    assert actual == expected, (
        f"Source fingerprint mismatch: {name}"
    )

checkpoint_module = ROOT / "scripts/reid_hf_checkpoints.py"

assert checkpoint_module.is_file(), (
    "Pinned research commit lacks "
    "scripts/reid_hf_checkpoints.py"
)

print("[PASS] Original research source fingerprints verified")


# ============================================================
# 7. Import original Hugging Face checkpoint manager
# ============================================================

sys.path.insert(0, str(ROOT))

from scripts.reid_hf_checkpoints import HuggingFaceCheckpointStore

from huggingface_hub import batch_bucket_files, bucket_info, create_bucket, download_bucket_files
print("[PASS] HF Bucket transport APIs available")


# ============================================================
# 8. Initialize FastReID after PyTorch/TorchVision preflight
# ============================================================

from reidfm.fastreid_compat import ensure_fastreid

ensure_fastreid()

print("[PASS] FastReID compatibility initialized")
print("[PASS] Checkpoint schema and pinned manager imported")


# ============================================================
# 9. Notebook-local HF checkpoint validator fix
# ============================================================

import scripts.reid_hf_checkpoints as hf_checkpoint_module


def inspect_actual_fastreid_checkpoint(
    path,
    *,
    iters_per_epoch,
    max_epoch,
    expected_run,
    expected_fingerprint,
):
    import torch

    from reidfm.engine.v3_mid_epoch_checkpoint import (
        resolve_v3_resume_cursor,
    )

    state = torch.load(
        str(path),
        map_location="cpu",
        weights_only=True,
    )

    if not isinstance(state, dict):
        raise ValueError(
            "Expected FastReID checkpoint dictionary"
        )

    required_states = (
        "model",
        "optimizer",
        "lr_sched",
        "warmup_sched",
        "v3_freeze_counter",
    )

    for name in required_states:
        if (
            name not in state
            or not isinstance(
                state[name],
                (dict, collections.OrderedDict),
            )
        ):
            raise ValueError(
                f"Checkpoint missing valid resumable state: {name}"
            )

    if path.name == "model_mid.pth":

        bundle = state.get("v3_multirank_bundle")

        if not isinstance(bundle, dict):
            raise ValueError(
                "Midpoint checkpoint missing two-rank resume bundle"
            )

        if (
            bundle.get("run_id") != expected_run
            or bundle.get("fingerprint") != expected_fingerprint
            or bundle.get("world_size") != 2
        ):
            raise ValueError(
                "Midpoint checkpoint identity/fingerprint/"
                "world-size mismatch"
            )

        epoch, within, completed = resolve_v3_resume_cursor(
            state,
            iters_per_epoch,
            max_epoch,
        )

    else:

        if (
            "v3_resume_schema" in state
            or "v3_multirank_bundle" in state
        ):
            raise ValueError(
                "Epoch checkpoint unexpectedly contains midpoint cursor"
            )

        epoch_index = state.get("epoch")

        if (
            type(epoch_index) is not int
            or not 0 <= epoch_index < max_epoch
        ):
            raise ValueError(
                "Invalid end-of-epoch checkpoint cursor"
            )

        epoch = epoch_index + 1
        within = 0
        completed = epoch * iters_per_epoch

    del state

    return {
        "epoch": epoch,
        "iteration_in_epoch": within,
        "global_steps": completed,
    }


hf_checkpoint_module.inspect_checkpoint = (
    inspect_actual_fastreid_checkpoint
)

print("[PASS] HF validator checks both real LR scheduler states")


# ============================================================
# 10. Final environment summary
# ============================================================

print()
print("=" * 65)
print(f"REID {CONDITION} / SEED {SEED} — ENVIRONMENT PREFLIGHT")
print("=" * 65)

print("[PASS] Research Git revision")
print("[PASS] FastReID Git revision")
print("[PASS] Required dependencies")
print("[PASS] PyTorch / TorchVision initialization")
print("[PASS] CUDA and 2 GPUs")
print("[PASS] Critical source fingerprints")
print("[PASS] FastReID compatibility")
print("[PASS] Verified checkpoint validator and HF Bucket APIs")
print("[PASS] Correct LR scheduler checkpoint validator")

print("=" * 65)
print("[READY] Continue to the next notebook cell")
print("=" * 65)

## 3. Training/validation manifests

Retrieve and validate the original study splits and their SHA256 hashes.


In [ ]:
import gdown

MANIFEST_DIR = ROOT / "unified/final_dataset/Global"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS = {
    "train_manifest.csv": (
        "15X1NxpVnkJGK_vMbqMxFD2GY0wh8WOCq",
        "40a5d4d66363c3300b801f8cf79b96cd84b4b44c9b22bf8d7f5e455bc89aa7a6"),
    "validation_manifest.csv": (
        "1-CTDEythFA9AgevNIGCv9Aniv3E-4Mr1",
        "55c3de1fe53960daef0a30e13961f926794beafc0f89bd4376261daa21630033"),
}
for name, (file_id, expected_sha) in MANIFESTS.items():
    path = MANIFEST_DIR / name
    if not path.is_file():
        with tempfile.TemporaryDirectory(dir="/kaggle/working") as tmp:
            staged = Path(tmp) / name
            result = gdown.download(id=file_id, output=str(staged), quiet=True)
            assert result and staged.is_file(), f"Cannot download {name}"
            assert hashlib.sha256(staged.read_bytes()).hexdigest() == expected_sha, name
            os.replace(staged, path)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected_sha, name
    print(f"[PASS] {name} verified (SHA-256)")

with (MANIFEST_DIR / "train_manifest.csv").open(encoding="utf-8") as fh:
    train_count = sum(1 for _ in fh) - 1
assert train_count == 782064, f"Unexpected training rows: {train_count}"
print("[PASS] 782,064 training records. Unseen evaluation targets are excluded.")


## 4. Verified fast PACK staging

Copy all ten PACKs to a stable temporary path, checking SHA256 without modifying source data.


In [ ]:
import hashlib
import json
import os
import shutil
import time

SOURCE_PACK_ROOT = PACK
LOCAL_PACK_DIR = Path("/kaggle/tmp/reid_v3_all_staged_packs")
HYBRID_PACK_ROOT = Path("/kaggle/tmp/reid_v3_all_local_pack_root")
PACK_STAGE_REPORT = Path("/kaggle/working/ReID_All_PACK_Tmp_Stage_Report.json")

STAGE_NAMES = tuple(sorted(p.name for p in SOURCE_PACK_ROOT.glob("*.pack")))

assert SOURCE_PACK_ROOT.is_dir()
source_packs = {p.name: p for p in SOURCE_PACK_ROOT.glob("*.pack")}
assert len(source_packs) == 10, f"Expected 10 mounted PACKs, found {len(source_packs)}"
assert set(STAGE_NAMES).issubset(source_packs), "Missing selected PACK files"

total_stage_bytes = sum(source_packs[name].stat().st_size for name in STAGE_NAMES)
margin_bytes = 8 * 1024**3
LOCAL_PACK_DIR.parent.mkdir(parents=True, exist_ok=True)
free_bytes = shutil.disk_usage("/kaggle/tmp").free
new_stage_bytes = sum(
    source_packs[name].stat().st_size
    for name in STAGE_NAMES
    if not (LOCAL_PACK_DIR / name).is_file()
)

print(f"[DISK] Free: {free_bytes / 1024**3:.2f} GiB")
print(f"[DISK] Planned stage: {total_stage_bytes / 1024**3:.2f} GiB")
print(f"[DISK] Additional space needed: {new_stage_bytes / 1024**3:.2f} GiB")
print(f"[DISK] Required reserve after staging: {margin_bytes / 1024**3:.2f} GiB")
assert free_bytes >= new_stage_bytes + margin_bytes, (
    "Not enough local disk for all selected PACKs plus 8 GiB reserve. "
    "Start a fresh Kaggle session or reduce the stage set."
)

LOCAL_PACK_DIR.mkdir(parents=True, exist_ok=True)
HYBRID_PACK_ROOT.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as inp:
        for block in iter(lambda: inp.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


staged = []
started = time.monotonic()

# Copy all 10 checked PACK files; never modify original files.
for name in STAGE_NAMES:
    original = source_packs[name]
    dest = LOCAL_PACK_DIR / name
    if dest.is_file():
        assert not dest.is_symlink(), f"Existing stage must be a local file: {dest}"
        assert dest.stat().st_size == original.stat().st_size, (
            f"Existing staged file has different size: {name}"
        )
        expected_hash = sha256_file(original)
        actual_hash = sha256_file(dest)
        assert actual_hash == expected_hash, f"Existing stage SHA mismatch: {name}"
        print(f"[REUSE] Verified existing stage: {name}")
    else:
        size_bytes = original.stat().st_size
        current_free = shutil.disk_usage("/kaggle/tmp").free
        assert current_free >= size_bytes + margin_bytes, (
            f"Not enough local disk to safely stage {name}; "
            f"need {(size_bytes + margin_bytes)/1024**3:.2f} GiB free, "
            f"have {current_free/1024**3:.2f} GiB. Use a fresh Kaggle session."
        )
        temporary = dest.with_name(name + ".partial")
        print(f"[STAGE] {name}: {size_bytes / 1024**3:.2f} GiB")
        source_hash = hashlib.sha256()
        with original.open("rb") as inp, temporary.open("wb") as out:
            while block := inp.read(8 * 1024 * 1024):
                source_hash.update(block)
                out.write(block)
            out.flush()
            os.fsync(out.fileno())
        assert temporary.stat().st_size == size_bytes, f"Incomplete copy: {name}"
        assert sha256_file(temporary) == source_hash.hexdigest(), (
            f"Local PACK copy failed SHA256: {name}"
        )
        os.replace(temporary, dest)
        expected_hash = source_hash.hexdigest()
        print(f"[PASS] SHA256 verified: {name}")
    staged.append({"name": name, "bytes": original.stat().st_size,
                   "sha256": expected_hash, "local_path": str(dest)})

# The model sees exactly 10 *.pack paths under one root; some are local,
# the rest point to the original immutable Kaggle input files.
for name, original in sorted(source_packs.items()):
    target = (LOCAL_PACK_DIR / name) if name in STAGE_NAMES else original
    link = HYBRID_PACK_ROOT / name
    if link.is_symlink():
        assert link.resolve() == target.resolve(), f"Unexpected previous link: {link}"
    elif link.exists():
        raise RuntimeError(f"Will not overwrite existing non-symlink: {link}")
    else:
        link.symlink_to(target)
    assert link.is_file(), f"Hybrid PACK path broken: {link}"

assert len(list(HYBRID_PACK_ROOT.glob("*.pack"))) == 10
PACK = HYBRID_PACK_ROOT  # The existing torchrun cell passes this as REID_PACK.
free_after = shutil.disk_usage("/kaggle/tmp").free
assert free_after >= margin_bytes, "Disk reserve unexpectedly exhausted"

stage_report = {
    "source_commit": RESEARCH_COMMIT,
    "source_pack_root": str(SOURCE_PACK_ROOT),
    "hybrid_pack_root": str(PACK),
    "staged_files": staged,
    "original_symlink_files": sorted(set(source_packs) - set(STAGE_NAMES)),
    "bytes_staged_total": total_stage_bytes,
    "free_gib_after_stage": round(free_after / 1024**3, 3),
    "stage_and_verification_wall_s": round(time.monotonic() - started, 3),
}
PACK_STAGE_REPORT.write_text(json.dumps(stage_report, indent=2) + "\n", encoding="utf-8")
print("[PASS] 10 hybrid PACK paths ready:", PACK)
print("[PASS] Exact local PACK byte integrity verified by SHA256")
print("[PASS] Free disk after staging:", round(free_after / 1024**3, 2), "GiB")


## 5. Experiment and fingerprint checks

Assert the original training config, seed, V3 protocol, and saved-checkpoint fingerprint.


In [ ]:
from reidfm.config import setup_cfg
from reidfm.experiments.loss_study import (
    load_loss_study_manifest, expand_loss_study_run,
)
from reidfm.experiments.protocol import config_fingerprint

study = load_loss_study_manifest()
run = expand_loss_study_run(CONDITION, SEED, study)
v3_opts = [
    "REIDFM.DATA_FORMAT", "REIDPACK3",
    "REIDFM.V3_TRAIN_MANIFEST", str(MANIFEST_DIR / "train_manifest.csv"),
    "REIDFM.V3_VAL_MANIFEST", str(MANIFEST_DIR / "validation_manifest.csv"),
    "REIDFM.V3_PACK_ROOT", str(PACK),
    "REIDFM.V3_PROTOCOL_ID", "dg_reid_global_p3_p5_v1",
    "REIDFM.STRICT_SPLIT", "True",
    "REIDFM.EVAL_AMP", "False",
    "MODEL.DEVICE", "cuda",
    "SOLVER.MAX_EPOCH", "50",
]
all_opts = list(run.opts) + v3_opts
cfg = setup_cfg(
    run.config_file, opts=all_opts, model_key=run.init_model,
    output_dir=str(OUT), prepare_weights=False, validate_study=True,
)
actual = (
    cfg.SOLVER.MAX_EPOCH, cfg.REIDFM.ITERS_PER_EPOCH,
    cfg.SOLVER.IMS_PER_BATCH, cfg.SOLVER.FREEZE_ITERS,
    cfg.SOLVER.CHECKPOINT_PERIOD, cfg.TEST.EVAL_PERIOD,
)
assert actual == (50, 2000, 128, 1000, 1, 5), f"Unexpected study protocol: {actual}"
assert cfg.REIDFM.DATA_FORMAT == "REIDPACK3"
assert cfg.MODEL.HEADS.NUM_CLASSES == 10036
fingerprint = config_fingerprint(cfg)
print(f"[PASS] {CONDITION} / Seed {SEED}, 50 epochs, 2000 steps, global batch 128")
print(f"[PASS] Config fingerprint: {fingerprint[:16]}…")


## 6. Verified HF Bucket checkpoint recovery

Auto-restore the last durable checkpoint, including optimizer/scheduler states; incompatible states fail safely.


In [ ]:
"""Notebook-embedded, non-versioned Hugging Face Bucket checkpoint transport.

This class is injected into each 50-epoch Kaggle notebook. The pinned
research/training implementation and its original FastReID checkpoint schema
remain unchanged. Only the remote checkpoint transport is replaced.

Remote heavy files per (condition, seed): slot_a.pth, slot_b.pth, model_best.pth.
No model repository uploads, Git commits, or versioned binary history.
"""

import json
import os
import tempfile
import time
from pathlib import Path

from huggingface_hub import (
    batch_bucket_files, bucket_info, create_bucket,
    download_bucket_files, list_bucket_tree,
)
from scripts.reid_hf_checkpoints import (
    HuggingFaceCheckpointStore, VALID_NAMES, SCHEMA, digest, safe_copy,
)
from scripts import reid_hf_checkpoints as hf_checkpoint_module


class RollingBucketCheckpointStore(HuggingFaceCheckpointStore):
    """Two rolling resume slots plus one best file (no Git history).

    The JSON pointer is written last. Uploading a replacement always targets
    the *other* resume slot so a failed transfer cannot replace the active one.
    """

    def __init__(self, *, bucket_id, **kwargs):
        super().__init__(repo_id=bucket_id, **kwargs)
        self.bucket_id = bucket_id
        self.prefix = (
            f"v3_50epoch_28c68a1/{self.condition}/seed{self.seed}"
        )

    def ensure_private_repository(self):
        """Retain the old call interface but ensure a private Bucket instead."""
        create_bucket(
            bucket_id=self.bucket_id, private=True,
            exist_ok=True, token=self.token,
        )
        info = bucket_info(self.bucket_id, token=self.token)
        if not info.private:
            raise RuntimeError("Refusing to save private ReID checkpoints to a public bucket")
        print("[PASS] Private HF Storage Bucket available (no Git history)")

    def _remote_keys(self):
        return {
            item.path for item in list_bucket_tree(
                self.bucket_id, prefix=self.prefix, recursive=True, token=self.token,
            ) if getattr(item, "type", "file") == "file"
        }

    def _read_json(self, key):
        if key not in self._remote_keys():
            return None
        with tempfile.TemporaryDirectory(dir=self.output) as td:
            dst = Path(td) / "metadata.json"
            download_bucket_files(
                self.bucket_id, files=[(key, dst)],
                raise_on_missing_files=True, token=self.token,
            )
            document = json.loads(dst.read_text(encoding="utf-8"))
        if not isinstance(document, dict):
            raise RuntimeError("Malformed HF Bucket metadata")
        return document

    def restore_latest(self):
        self.output.mkdir(parents=True, exist_ok=True)
        doc = self._read_json(self._key("latest.json"))
        if doc is not None:
            self._validate_identity(doc)
        pointer = self.output / "last_checkpoint"
        if pointer.is_file():
            name = pointer.read_text(encoding="utf-8").strip()
            if name not in VALID_NAMES or not (self.output / name).is_file():
                raise RuntimeError("Unsafe or incomplete local resume pointer")
            cursor = hf_checkpoint_module.inspect_checkpoint(
                self.output / name, iters_per_epoch=self.iters_per_epoch,
                max_epoch=self.max_epoch,
                expected_run=f"{self.condition}/seed_{self.seed}",
                expected_fingerprint=self.fingerprint,
            )
            if doc is None or cursor["global_steps"] >= doc["global_steps"]:
                self.latest = doc
                return {**(doc or {}), **cursor, "source": "local"}
        if doc is None:
            return None
        with tempfile.TemporaryDirectory(dir=self.output) as td:
            downloaded = Path(td) / doc["filename"]
            download_bucket_files(
                self.bucket_id,
                files=[(doc["remote_file"], downloaded)],
                raise_on_missing_files=True, token=self.token,
            )
            if downloaded.stat().st_size != doc["size"] or digest(downloaded) != doc["sha256"]:
                raise RuntimeError("Remote checkpoint checksum mismatch; refusing unsafe resume")
            cursor = hf_checkpoint_module.inspect_checkpoint(
                downloaded, iters_per_epoch=self.iters_per_epoch,
                max_epoch=self.max_epoch,
                expected_run=f"{self.condition}/seed_{self.seed}",
                expected_fingerprint=self.fingerprint,
            )
            if cursor["global_steps"] != doc["global_steps"]:
                raise RuntimeError("Remote checkpoint cursor/manifest mismatch")
            target = self.output / doc["filename"]
            safe_copy(downloaded, target)
        temporary = pointer.with_name("last_checkpoint.tmp")
        temporary.write_text(doc["filename"], encoding="utf-8")
        os.replace(temporary, pointer)
        self.latest = doc
        return doc

    def publish_latest(self):
        pointer = self.output / "last_checkpoint"
        if not pointer.is_file():
            return None
        filename = pointer.read_text(encoding="utf-8").strip()
        if filename not in VALID_NAMES or not (self.output / filename).is_file():
            return None
        with tempfile.TemporaryDirectory(dir=self.output) as td:
            staged = Path(td) / filename
            safe_copy(self.output / filename, staged, marker=pointer)
            cursor = hf_checkpoint_module.inspect_checkpoint(
                staged, iters_per_epoch=self.iters_per_epoch,
                max_epoch=self.max_epoch,
                expected_run=f"{self.condition}/seed_{self.seed}",
                expected_fingerprint=self.fingerprint,
            )
            if self.latest and cursor["global_steps"] <= self.latest["global_steps"]:
                return None
            slot = (
                "slot_b.pth" if self.latest and
                self.latest["remote_file"].endswith("slot_a.pth")
                else "slot_a.pth"
            )
            destination = self._key(slot)
            doc = {
                **self._base_metadata(), **cursor,
                "filename": filename, "remote_file": destination,
                "sha256": digest(staged), "size": staged.stat().st_size,
                "saved_at_unix": int(time.time()),
            }
            # Non-versioned overwrite of the INACTIVE slot first.
            batch_bucket_files(
                self.bucket_id, add=[(staged, destination)], token=self.token,
            )
            # Publish authoritative pointer only when upload succeeded.
            batch_bucket_files(
                self.bucket_id,
                add=[(json.dumps(doc, sort_keys=True).encode(), self._key("latest.json"))],
                token=self.token,
            )
        self.latest = doc
        return doc

    def publish_best(self):
        source = self.output / "model_best.pth"
        if not source.is_file():
            return False
        signature = (source.stat().st_size, source.stat().st_mtime_ns)
        if signature == self.best_signature:
            return False
        with tempfile.TemporaryDirectory(dir=self.output) as td:
            staged = Path(td) / source.name
            safe_copy(source, staged)
            if signature != (source.stat().st_size, source.stat().st_mtime_ns):
                raise RuntimeError("Best model changed during snapshot; retry later")
            import torch
            state = torch.load(str(staged), map_location="cpu", weights_only=True)
            doc = {
                **self._base_metadata(), "filename": source.name,
                "sha256": digest(staged), "size": staged.stat().st_size,
                "best_epoch_index": int(state.get("epoch", -1)),
                "validation_metric": float(state.get("metric", -1.0)),
                "saved_at_unix": int(time.time()),
            }
            del state
            batch_bucket_files(
                self.bucket_id,
                add=[(staged, self._key("model_best.pth"))], token=self.token,
            )
            batch_bucket_files(
                self.bucket_id,
                add=[(json.dumps(doc, sort_keys=True).encode(), self._key("best.json"))],
                token=self.token,
            )
        self.best_signature = signature
        return True

    def publish_metrics(self, force=False):
        source = self.output / "metrics.json"
        if not source.is_file():
            return False
        signature = (source.stat().st_size, source.stat().st_mtime_ns)
        if not force and signature == self.metrics_signature:
            return False
        with tempfile.TemporaryDirectory(dir=self.output) as td:
            staged = Path(td) / "metrics.json"
            # Metrics are small JSON, unlike .pth weights (safe_copy requires
            # a >=1 KiB checkpoint). Guard against concurrent edits instead.
            staged.write_bytes(source.read_bytes())
            if signature != (source.stat().st_size, source.stat().st_mtime_ns):
                raise RuntimeError("Metrics changed during snapshot; retry later")
            batch_bucket_files(
                self.bucket_id,
                add=[(staged, self._key("sessions/metrics.json"))],
                token=self.token,
            )
        self.metrics_signature = signature
        return True

    def audit_storage(self):
        """Fail if unexpected large objects accumulate in this run prefix."""
        entries = list(list_bucket_tree(
            self.bucket_id, prefix=self.prefix, recursive=True, token=self.token,
        ))
        weights = [
            item for item in entries
            if item.path.endswith(".pth") and
            item.path.startswith(self.prefix + "/")
        ]
        allowed = {
            self._key("slot_a.pth"), self._key("slot_b.pth"),
            self._key("model_best.pth"),
        }
        unexpected = [item.path for item in weights if item.path not in allowed]
        if unexpected or len(weights) > 3:
            raise RuntimeError(f"Unexpected Bucket weight files: {unexpected}")
        total = sum(int(getattr(item, "size", 0)) for item in weights)
        return {"weight_files": len(weights), "weight_bytes": total}


store = RollingBucketCheckpointStore(
    bucket_id=HF_BUCKET_ID, token=HF_TOKEN, condition=CONDITION, seed=SEED,
    research_commit=RESEARCH_COMMIT, fingerprint=fingerprint, output=OUT,
    max_epoch=50, iters_per_epoch=2000,
    session_tag=str(int(SESSION_START)),
)
assert store.prefix == f"v3_50epoch_28c68a1/{CONDITION}/seed{SEED}"
print("[HF BUCKET] Isolated, non-versioned checkpoint prefix:", store.prefix)
store.ensure_private_repository()
resume_state = store.restore_latest()
if resume_state:
    print("[RESUME] Verified Bucket checkpoint at step:", resume_state["global_steps"])
else:
    print("[FRESH] No 50-epoch Bucket pointer; starting at ImageNet init")


## 7. Prepare production DDP worker

Original training/evaluation hooks plus notebook-local DDP config synchronization and event messages.

The DDP process-group timeout is expanded to 2 hours for rank-0-only validation, preserving the original optimizer and evaluator.


In [ ]:
PILOT_STYLE_WORKER = '"""Production B0/Seed2 worker: pinned 2-GPU torchrun training."""\nimport collections\nimport collections.abc\nimport json\nimport os\nimport statistics\nimport sys\nimport time\nimport traceback\nfrom collections import deque\nfrom pathlib import Path\n\nimport torch\nimport torch.distributed as dist\n\nfor name in ("Mapping", "MutableMapping", "Sequence", "Iterable", "Callable"):\n    if not hasattr(collections, name):\n        setattr(collections, name, getattr(collections.abc, name))\n\nROOT = Path(os.environ["REID_ROOT"])\nOUTPUT = Path(os.environ["REID_OUT"])\nsys.path.insert(0, str(ROOT))\n\nfrom reidfm.fastreid_compat import ensure_fastreid\nensure_fastreid()\nfrom fastreid.engine import hooks\nfrom fastreid.utils import comm\nfrom fastreid.utils.env import seed_all_rng\nfrom fastreid.utils.events import get_event_storage\nfrom reidfm.config import setup_cfg\nfrom reidfm.engine.trainer import ReidfmTrainer\nfrom reidfm.engine.v3_trainer_midresume import V3TrainerMidpointHook\nfrom reidfm.experiments.loss_study import load_loss_study_manifest, expand_loss_study_run\nfrom reidfm.experiments.protocol import config_fingerprint\n\nrank = int(os.environ["RANK"])\ndevice = int(os.environ["LOCAL_RANK"])\nresuming = os.environ["REID_RESUME"] == "1"\nprocess_start = time.monotonic()\n\n\ndef report(status, **details):\n    payload = {\n        "rank": rank, "status": status,\n        "process_elapsed_s": round(time.monotonic() - process_start, 2),\n        "time_unix": time.time(), **details,\n    }\n    path = OUTPUT / f"rank{rank}_live.json"\n    temporary = OUTPUT / f"rank{rank}_live.json.tmp"\n    temporary.write_text(json.dumps(payload), encoding="utf-8")\n    os.replace(temporary, path)\n    if status != "training" or (details.get("iteration", 0) % 500 == 0):\n        print("[WORKER]", json.dumps(payload), flush=True)\n\n\nclass TimedInput:\n    """Observe the existing DataLoader iterator, with no sampling changes."""\n    def __init__(self, original):\n        self.original = original\n        self.recent_waits = deque(maxlen=100)\n\n    def __iter__(self):\n        return self\n\n    def __next__(self):\n        started = time.monotonic()\n        item = next(self.original)\n        self.recent_waits.append(time.monotonic() - started)\n        return item\n\n\nclass PilotStyleProgress(hooks.HookBase):\n    """Pilot\'s after_step telemetry, without changing forward/backward or DDP."""\n    def __init__(self, timed_input):\n        self.timed_input = timed_input\n        self.previous_amp_scale = None\n        self.amp_drops = 0\n        self.last_step_time = None\n        self.recent_step_durations = deque(maxlen=100)\n\n    def before_epoch(self):\n        # Midpoint resume can replay 1000 batches before before_epoch; do not\n        # confuse those replay reads with training input-wait measurements.\n        self.timed_input.recent_waits.clear()\n        self.recent_step_durations.clear()\n        self.last_step_time = None\n\n    def after_step(self):\n        current = time.monotonic()\n        if self.last_step_time is not None:\n            self.recent_step_durations.append(current - self.last_step_time)\n        self.last_step_time = current\n\n        step = int(self.trainer.iter) + 1\n        epoch = int(self.trainer.epoch) + 1\n        scaler = getattr(self.trainer._trainer, "grad_scaler", None)\n        scale = float(scaler.get_scale()) if scaler is not None else None\n        if scale is not None and self.previous_amp_scale is not None:\n            if scale < self.previous_amp_scale:\n                self.amp_drops += 1\n        self.previous_amp_scale = scale\n\n        should_report = step == 1 or step % 25 == 0\n        if not should_report:\n            return\n\n        losses = {}\n        if rank == 0 and step % 100 == 0:\n            for name, (value, _) in get_event_storage().latest().items():\n                if "loss" in name.lower():\n                    losses[name] = round(float(value), 4)\n        recent = (statistics.mean(self.recent_step_durations)\n                  if self.recent_step_durations else None)\n        waits = list(self.timed_input.recent_waits)\n        report(\n            "training", iteration=step, epoch=epoch,\n            completed_in_epoch=(step - 1) % 2000 + 1,\n            recent_seconds_per_step=round(recent, 4) if recent is not None else None,\n            recent_steps_per_second=round(1 / recent, 4) if recent else None,\n            recent_data_wait_s=round(statistics.mean(waits), 4) if waits else None,\n            data_wait_events_over_100ms=sum(value > 0.1 for value in waits),\n            amp_scale=scale, amp_scale_drops=self.amp_drops,\n            losses=losses,\n            peak_vram_gib=round(torch.cuda.max_memory_allocated(device) / 2**30, 2),\n        )\n\n\ntry:\n    torch.cuda.set_device(device)\n    # The full V3 evaluation executes on rank0 while rank1 waits in the\n    # FastReID EvalHook barrier. The default 10-minute NCCL watchdog deadline\n    # killed the previous run at step 10,000; allow up to 2h of evaluation.\n    # This changes only communication timeout, not model, data or metrics.\n    from datetime import timedelta\n    dist.init_process_group(\n        backend="nccl", device_id=torch.device("cuda", device),\n        timeout=timedelta(hours=2),\n    )\n    assert dist.get_world_size() == 2\n    comm._LOCAL_PROCESS_GROUP = dist.new_group(ranks=[0, 1], backend="nccl")\n    assert comm.get_local_rank() == device\n    report("gpu_ready", gpu_name=torch.cuda.get_device_name(device))\n\n    run = expand_loss_study_run(os.environ["REID_CONDITION"], int(os.environ["REID_SEED"]), load_loss_study_manifest())\n    opts = list(run.opts) + [\n        "REIDFM.DATA_FORMAT", "REIDPACK3",\n        "REIDFM.V3_TRAIN_MANIFEST", str(ROOT / "unified/final_dataset/Global/train_manifest.csv"),\n        "REIDFM.V3_VAL_MANIFEST", str(ROOT / "unified/final_dataset/Global/validation_manifest.csv"),\n        "REIDFM.V3_PACK_ROOT", os.environ["REID_PACK"],\n        "REIDFM.V3_PROTOCOL_ID", "dg_reid_global_p3_p5_v1",\n        "REIDFM.STRICT_SPLIT", "True",\n        "REIDFM.EVAL_AMP", "False",\n        "MODEL.DEVICE", "cuda",\n        "SOLVER.MAX_EPOCH", "50",\n    ]\n    import fcntl\n    # setup_cfg writes a shared temporary YAML overlay in the pinned checkout.\n    # Protect against rank0/rank1 concurrent reads/writes; does not change cfg.\n    with (OUTPUT / "ddp_config_overlay.lock").open("a+") as setup_guard:\n        fcntl.flock(setup_guard.fileno(), fcntl.LOCK_EX)\n        try:\n            cfg = setup_cfg(\n                run.config_file, opts=opts, model_key=run.init_model,\n                output_dir=str(OUTPUT), prepare_weights=not resuming,\n                time_budget_h=11.0, validate_study=True,\n            )\n        finally:\n            fcntl.flock(setup_guard.fileno(), fcntl.LOCK_UN)\n    assert config_fingerprint(cfg) == os.environ["REID_EXPECTED_FINGERPRINT"], (\n        "Scientific fingerprint mismatch between notebook and worker"\n    )\n    assert (\n        cfg.SOLVER.MAX_EPOCH, cfg.REIDFM.ITERS_PER_EPOCH,\n        cfg.SOLVER.FREEZE_ITERS, cfg.SOLVER.IMS_PER_BATCH,\n        cfg.TEST.EVAL_PERIOD, cfg.DATALOADER.NUM_WORKERS,\n    ) == (50, 2000, 1000, 128, 5, 4)\n    # Operational checkpoint policy, intentionally excluded from the pinned\n    # training fingerprint. Does not modify training math or sampler schedule.\n    cfg.defrost()\n    cfg.REIDFM.MID_EPOCH_CHECKPOINT = os.environ.get("REID_MID_EPOCH_CHECKPOINT") == "1"\n    cfg.freeze()\n    seed_all_rng(int(cfg.REIDFM.SEED) + rank)\n\n    report("building_trainer")\n    trainer = ReidfmTrainer(cfg)\n    assert trainer.max_epoch == 50\n    assert trainer._trainer.data_loader.dataset.num_classes == 10036\n    assert trainer._v3_freeze_counter is not None\n    # Keep support for loading older midpoint checkpoints, but optionally\n    # disable future midpoint writes. End-of-epoch checkpointing remains on.\n    if not cfg.REIDFM.MID_EPOCH_CHECKPOINT:\n        trainer._hooks = [\n            h for h in trainer._hooks\n            if not isinstance(h, V3TrainerMidpointHook)\n        ]\n        assert not any(isinstance(h, V3TrainerMidpointHook) for h in trainer._hooks)\n    else:\n        assert any(isinstance(h, V3TrainerMidpointHook) for h in trainer._hooks)\n    if rank == 0:\n        assert any(isinstance(h, hooks.EvalHook) for h in trainer._hooks), (\n            "Production validation hook must NOT be disabled"\n        )\n    # Monitoring only: preserve the original EvalHook execution and return.\n    for eval_hook in trainer._hooks:\n        if isinstance(eval_hook, hooks.EvalHook):\n            original_func = eval_hook._func\n            def monitored_eval(_func=original_func):\n                if rank == 0:\n                    print(f"[STAGE][EVAL] Starting evaluation — epoch {trainer.epoch + 1}", flush=True)\n                    print("[STAGE][EVAL] Extracting features and computing retrieval metrics", flush=True)\n                started = time.monotonic()\n                result = _func()\n                if rank == 0:\n                    print(f"[STAGE][EVAL] Evaluation completed in {time.monotonic()-started:.1f}s", flush=True)\n                return result\n            eval_hook._func = monitored_eval\n    timed_input = TimedInput(trainer._trainer._data_loader_iter)\n    trainer._trainer._data_loader_iter = timed_input\n    trainer.register_hooks([PilotStyleProgress(timed_input)])\n    trainer.resume_or_load(resume=resuming)\n    report("trainer_ready", resume=resuming, start_epoch=trainer.start_epoch,\n           resumed_offset=int(getattr(trainer, "_v3_resume_offset", 0)),\n           max_epochs=trainer.max_epoch)\n    dist.barrier()\n    report("training_started")\n\n    trainer.train()  # ORIGINAL 50-epoch loop; all production hooks retained.\n    dist.barrier()\n\n    completed = int(trainer.iter)\n    assert 0 < completed <= 50 * 2000\n    finished = {\n        "rank": rank, "global_steps": completed,\n        "finished_all_50_epochs": completed == 50 * 2000,\n        "stopped_cleanly_at_epoch_boundary": completed % 2000 == 0,\n        "peak_gpu_memory_gib": round(torch.cuda.max_memory_allocated(device) / 2**30, 2),\n        "freeze_count": trainer._v3_freeze_counter.count,\n    }\n    combined = [None, None]\n    dist.all_gather_object(combined, finished)\n    if rank == 0:\n        assert all(item["global_steps"] == completed for item in combined)\n        last_checkpoint = OUTPUT / "last_checkpoint"\n        assert last_checkpoint.is_file(), "No durable local checkpoint pointer"\n        final = {\n            "success": True, "condition": os.environ["REID_CONDITION"], "seed": int(os.environ["REID_SEED"]),\n            "study_epochs": 50, "steps_per_epoch": 2000, "world_size": 2,\n            "global_batch_size": 128, "evaluation_enabled": True,\n            "resume_mode": resuming,\n            "global_steps_completed": completed,\n            "last_checkpoint": last_checkpoint.read_text().strip(),\n            "ranks": combined,\n        }\n        (OUTPUT / "pilot_style_session_report.json").write_text(\n            json.dumps(final, indent=2) + "\\n", encoding="utf-8"\n        )\n    report("finished", **finished)\nexcept BaseException as exc:\n    report("error", message=str(exc)[:1600], traceback=traceback.format_exc()[-5000:])\n    raise\nfinally:\n    if dist.is_initialized():\n        dist.destroy_process_group()\n'

from pathlib import Path
import ast
import textwrap

WORKER_FILE = OUT / f"{CONDITION.lower()}_seed{SEED}_pilot_style_worker.py"
WORKER_FILE.write_text(textwrap.dedent(PILOT_STYLE_WORKER).strip() + "\n", encoding="utf-8")
ast.parse(WORKER_FILE.read_text(encoding="utf-8"))
print("[PASS] Pilot-style torchrun worker ready: two DDP ranks / two GPUs")
print("[PASS] Pinned loss-study training algorithm unchanged; Original training and evaluation retained; midpoint save policy configurable")
print("[FILE]", WORKER_FILE)


## 8. Training and live stage monitoring

Training steps advance the bar; validation, local checkpoint saves, and HF Bucket rolling uploads print separate stage events.


In [ ]:
import json
import os
import signal
import statistics
import subprocess
import sys
import threading
import time
from collections import deque
from pathlib import Path
from tqdm.auto import tqdm

LOG = OUT / "kaggle_training_console.log"
REPORT = OUT / "pilot_style_session_report.json"
assert WORKER_FILE.is_file(), "Run worker preparation cell first"
# These are only live status counters, never checkpoints. Remove stale
# counters so a previous Kaggle attempt cannot make a fresh run look faster.
for transient in (OUT / "rank0_live.json", OUT / "rank1_live.json"):
    transient.unlink(missing_ok=True)
session_report_cutoff_unix = time.time()

# The model always trains on both GPUs. Resuming is verified by the existing
# RollingBucketCheckpointStore restore step, not a guessed local checkpoint.
training_env = os.environ.copy()
training_env.update({
    "REID_ROOT": str(ROOT), "REID_OUT": str(OUT),
    "REID_PACK": str(PACK), "REID_RESUME": "1" if resume_state else "0",
    "REID_EXPECTED_FINGERPRINT": fingerprint,
    "REID_CONDITION": CONDITION,
    "REID_SEED": str(SEED),
    "REID_MID_EPOCH_CHECKPOINT": "1" if ENABLE_MID_EPOCH_CHECKPOINT else "0",
    "CUDA_VISIBLE_DEVICES": "0,1", "PYTHONUNBUFFERED": "1",
    "PYTHONPATH": os.pathsep.join(filter(None, [str(ROOT), training_env.get("PYTHONPATH", "")])),
})
command = [sys.executable, "-m", "torch.distributed.run",
           "--standalone", "--nnodes=1", "--nproc_per_node=2", str(WORKER_FILE)]

# The rolling HF Bucket manager does not touch the model, optimizer, batch sampler
# or process group. This optional background thread publishes verified files
# asynchronously; it never makes optimizer steps or changes the loss-study condition.
events = deque(maxlen=20)
errors = deque(maxlen=10)
backup_stop = threading.Event()

def backup_worker():
    while not backup_stop.is_set():
        try:
            if (OUT / "last_checkpoint").is_file():
                newest = store.publish_latest()
                if newest:
                    events.append(f"[HF BUCKET] Backup verified — step {newest['global_steps']:,}")
                    store.publish_metrics(force=True)
                if (OUT / "model_best.pth").is_file() and store.publish_best():
                    events.append("[HF BUCKET] Best checkpoint uploaded")
                errors.clear()
        except Exception as exc:
            errors.append(f"{type(exc).__name__}: {exc}")
        backup_stop.wait(25)
    for attempt in range(6):
        try:
            newest = store.publish_latest()
            if newest:
                events.append(f"[HF BUCKET] Final backup verified — step {newest['global_steps']:,}")
            store.publish_best()
            store.publish_metrics(force=True)
            errors.clear()
            return
        except Exception as exc:
            if attempt == 5:
                errors.append(f"Final sync: {type(exc).__name__}: {exc}")
            else:
                time.sleep(5)


last_progress = int(resume_state["global_steps"]) if resume_state else 0
completed_report = None
training_start = time.monotonic()
# Read subprocess events without printing the whole verbose worker log.
log_offset = LOG.stat().st_size if LOG.exists() and resume_state else 0
seen_training_epochs = set()
last_checkpoint_name = None

def show_stage_events():
    global log_offset
    if not LOG.is_file():
        return
    with LOG.open("r", encoding="utf-8", errors="replace") as handle:
        handle.seek(log_offset)
        lines = handle.readlines()
        log_offset = handle.tell()
    for line in lines:
        if "[STAGE]" in line:
            tqdm.write(line.split("[STAGE]", 1)[1].strip())
        elif "V3 val " in line and "R1 " in line and "mAP " in line:
            tqdm.write("[EVAL] " + line.split("V3 val ", 1)[1].strip())
        elif "[WORKER]" in line and '"status": "error"' in line:
            tqdm.write("[ERROR] Worker reported failure — see training log")
backup_thread = threading.Thread(target=backup_worker, daemon=True, name="hf-safe-backup")
backup_thread.start()
job = None
log_handle = LOG.open("a" if resume_state else "w", encoding="utf-8")

print(f"[START] Original {CONDITION}/Seed{SEED} training, 2 GPUs, global batch 128", flush=True)
print("[START] Same direct torchrun launch and after_step telemetry as the successful pilot", flush=True)
print("[CONFIG] 50 epochs × 2000 steps, eval every 5 epochs, exact resume, HF Bucket backup", flush=True)
try:
    job = subprocess.Popen(command, cwd=str(ROOT), env=training_env,
                           stdout=log_handle, stderr=subprocess.STDOUT,
                           start_new_session=True)
    with tqdm(total=100000, initial=last_progress,
              desc=f"{CONDITION} / Seed {SEED} | 2×T4", unit="step",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} "
                         "[monitor {elapsed}, {postfix}]") as bar:
        while job.poll() is None:
            status = {}
            status_path = OUT / "rank0_live.json"
            if status_path.is_file():
                try:
                    status = json.loads(status_path.read_text(encoding="utf-8"))
                except (OSError, ValueError):
                    status = {}
                if status.get("status") == "error":
                    raise RuntimeError("Worker error: " + status.get("message", "unknown"))
                if status.get("status") == "training":
                    steps = int(status.get("iteration", last_progress))
                    if steps > last_progress:
                        bar.update(min(100000, steps) - last_progress)
                        last_progress = min(100000, steps)
                        speed = status.get("recent_steps_per_second")
                        if speed:
                            bar.set_postfix_str(
                                f"ACTUAL recent={speed:.2f} step/s; "
                                f"{CONDITION} data wait={status.get('recent_data_wait_s')}s; "
                                f"AMP={status.get('amp_scale')}; "
                                f"drops={status.get('amp_scale_drops')}"
                            )
            show_stage_events()
            if status.get("status") == "training":
                epoch = int(status.get("epoch", 0))
                if epoch > 0 and epoch not in seen_training_epochs:
                    seen_training_epochs.add(epoch)
                    tqdm.write(f"[TRAIN] Epoch {epoch} running — step {last_progress:,}")
            checkpoint_pointer = OUT / "last_checkpoint"
            if checkpoint_pointer.is_file():
                try:
                    new_name = checkpoint_pointer.read_text(encoding="utf-8").strip()
                    if new_name and new_name != last_checkpoint_name:
                        last_checkpoint_name = new_name
                        tqdm.write(f"[CHECKPOINT] Local checkpoint recorded: {new_name}")
                except OSError:
                    pass
            while events:
                tqdm.write(events.popleft())
            if errors:
                tqdm.write("[HF BUCKET RETRY] " + errors[-1])
                errors.clear()
            time.sleep(2)
        exit_code = job.wait(timeout=20)
        if exit_code == 0 and (OUT / "rank0_live.json").is_file():
            final_live = json.loads((OUT / "rank0_live.json").read_text())
            if final_live.get("status") == "finished":
                step = int(final_live["global_steps"])
                if step > last_progress:
                    bar.update(min(100000, step) - last_progress)
                    last_progress = min(100000, step)
except KeyboardInterrupt:
    print("[STOP] Interrupt received; terminating ONLY the experiment workers")
    raise
finally:
    if job is not None and job.poll() is None:
        try:
            os.killpg(job.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            job.wait(timeout=20)
        except subprocess.TimeoutExpired:
            os.killpg(job.pid, signal.SIGKILL)
            job.wait(timeout=20)
    backup_stop.set()
    backup_thread.join(timeout=900)
    log_handle.close()

if backup_thread.is_alive():
    raise RuntimeError("HF final checkpoint sync is still running; inspect before ending session")
show_stage_events()
while events:
    print(events.popleft())
if errors:
    raise RuntimeError("Hugging Face Bucket backup failed: " + errors[-1])
if (job is None or job.returncode != 0 or not REPORT.is_file()
        or REPORT.stat().st_mtime < session_report_cutoff_unix):
    print("\n".join(LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-60:]))
    raise RuntimeError("Training process failed or did not write a session report")
if not store.latest:
    raise RuntimeError("No checkpoint was durably backed up to Hugging Face")

completed_report = json.loads(REPORT.read_text(encoding="utf-8"))
assert completed_report["success"]
completed_steps = int(completed_report["global_steps_completed"])
assert int(store.latest["global_steps"]) >= completed_steps, (
    "Newest local checkpoint not yet durable on Hugging Face"
)
print("[PASS] Both GPU ranks finished cleanly; model checkpoint backed up")
print("[STEPS]", completed_steps, "/", 100000)
print("[STATUS]", "ALL 50 EPOCHS COMPLETE" if completed_steps == 100000
      else "PAUSED SAFELY — rerun the notebook to resume")
print("[HF BUCKET] latest durable global step", store.latest["global_steps"])
print("[LOG]", LOG)
print("[REPORT]", REPORT)

print("[HF BUCKET] Live weight storage audit:", store.audit_storage())


## 9. Verify session state

Inspect completed steps and successful remote backup. Partial runs are not final scientific results.


In [ ]:
import json
from pathlib import Path

report_path = OUT / "pilot_style_session_report.json"
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("[STATUS]", "PASS" if report["success"] else "INCOMPLETE")
    print("[GLOBAL STEPS]", report["global_steps_completed"], "/", 100000)
    print("[LAST CHECKPOINT]", report["last_checkpoint"])
    print("[RANKS]", report["ranks"])
else:
    print("[PENDING] No completed training session yet. Read kaggle_training_console.log")
print("[OUTPUT]", OUT)


## 10. Export light diagnostics

Save logs and status reports in a small archive; model weights remain on the private HF Bucket.


In [ ]:
import zipfile

RESULT_ZIP = Path(f"/kaggle/working/ReID_{CONDITION}_Seed{SEED}_2GPU_50Epoch_Session_Results.zip")
included = []
with zipfile.ZipFile(RESULT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for local, name in (
        (PACK_STAGE_REPORT, "all_pack_tmp_stage_report.json"),
        (OUT / "kaggle_training_console.log", "kaggle_training_console.log"),
        (OUT / "pilot_style_session_report.json", "production_session_report.json"),
        (OUT / "rank0_live.json", "rank0_live.json"),
        (OUT / "rank1_live.json", "rank1_live.json"),
        (OUT / "metrics.json", "metrics.json"),
    ):
        if local.is_file():
            archive.write(local, arcname=name)
            included.append(name)

assert "all_pack_tmp_stage_report.json" in included
print("[RESULTS]", RESULT_ZIP)
print("[FILES]", ", ".join(included))
print("[NOTE] Model checkpoints are preserved via Hugging Face, not included here.")
